In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import requests
import mplfinance as mpf
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from lightweight_charts import Chart

In [8]:
#ISL Sweep+BOS Signal - Dynamic (Scored)

Ticker_df = pd.read_csv('NSE_Symbols.csv')

def telegram_alert(message):
    url = f'https://api.telegram.org/bot8520982329:AAGr_PfQPzUFI2zQDWJ41TAtmCobla2JcZY/sendMessage'
    requests.post(url, data={'chat_id': '-5245957606', 'text': message})

def mean_atr(df):
    prev_c = df['Close'].shift(1).fillna(df['Close'].iloc[0])
    tr = pd.concat([
        df['High'] - df['Low'],
        (df['High'] - prev_c).abs(),
        (df['Low']  - prev_c).abs()
    ], axis=1).max(axis=1)
    return tr.mean()

ATR_BASELINE = 14    # candles ending AT swing high used as volatility baseline (before downswing)
position     = -265
period       = round(((1/7) * -position) + 10)

# Alert_List tuple: (Ticker, depth_ratio, atr_ratio, fvg_atr, isl_buffer_pct, score, alert_time)
#   depth_ratio    = leg_drop / atr_baseline          — sweep depth in ATR units
#   atr_ratio      = mean_ATR(downswing) / atr_base   — candle energy during the drop
#   fvg_atr        = total_bearish_FVG_size / atr_base — gap cleanliness on the drop
#   isl_buffer_pct = (prev_SL_above_ISL - ISL) / ISL × 100  — display only, not in score
#   score          = depth_ratio × atr_ratio × (1 + fvg_atr)
Alert_List   = []
scan_count   = 0
signal_count = 0

for Ticker in Ticker_df["Symbol"]:
    BTC = yf.Ticker(Ticker).history(period=f'{period}d', interval='1h')

    if BTC is None or BTC.empty:
        continue
    if len(BTC) <= abs(position):
        continue

    scan_count += 1

    # ── Swing detection ───────────────────────────────────────────────────────
    BTC['Master_Swing'] = np.select(
        [(BTC['High'] > BTC['High'].shift(-1)) & (BTC['High'] > BTC['High'].shift(1)),
         (BTC['Low']  < BTC['Low'].shift(-1))  & (BTC['Low']  < BTC['Low'].shift(1))],
        ["Swing High", "Swing Low"], default="")

    BTC["Candle No."] = range(len(BTC), 0, -1)
    Subset_df = BTC[BTC['Master_Swing'] != ''].copy()

    # ── Slope / Price Delta ───────────────────────────────────────────────────
    Subset_df["Slope"] = np.select(
        [(Subset_df['Master_Swing'] == "Swing Low")  & (Subset_df['Master_Swing'].shift(1) == "Swing High"),
         (Subset_df['Master_Swing'] == "Swing High") & (Subset_df['Master_Swing'].shift(1) == "Swing Low")],
        ["Sell Slope", "Buy Slope"], default='')

    Subset_df["Price Delta"] = np.select(
        [(Subset_df["Slope"] == "Sell Slope"),
         (Subset_df["Slope"] == "Buy Slope")],
        [(Subset_df["High"].shift(1) - Subset_df["Low"]) /
         (Subset_df["Candle No."].shift(1) - Subset_df["Candle No."]),
         (Subset_df["High"] - Subset_df["Low"]).shift(1) /
         (Subset_df["Candle No."].shift(1) - Subset_df["Candle No."])],
        default=np.nan)

    BTC.loc[Subset_df.index, "Slope"]       = Subset_df["Slope"]
    BTC.loc[Subset_df.index, "Price Delta"] = Subset_df["Price Delta"]

    # ── ATR (rolling, for reference columns) ─────────────────────────────────
    BTC['H-L'] = BTC["High"]  - BTC["Low"]
    BTC['H-C'] = BTC["High"]  - BTC["Close"].shift(1)
    BTC['L-C'] = BTC["Low"]   - BTC["Close"].shift(1)
    BTC["TR"]  = BTC[['H-L', 'H-C', 'L-C']].max(axis=1)
    BTC["ATR"] = BTC["TR"].rolling(14).mean()

    Subset_ATR = BTC[BTC['Master_Swing'] != ''].copy()
    Subset_ATR["Impluse Strength"] = Subset_ATR["Price Delta"] / Subset_ATR["ATR"].shift(1)
    BTC.loc[Subset_ATR.index, "Impluse Strength"] = Subset_ATR["Impluse Strength"]

    # ── Intermediate Swings (ISL / ISH) ───────────────────────────────────────
    swing_lows  = Subset_df.loc[Subset_df["Master_Swing"] == "Swing Low",  ["Low"]].copy()
    swing_highs = Subset_df.loc[Subset_df["Master_Swing"] == "Swing High", ["High"]].copy()

    swing_lows["Intermediate_Swing_Low"] = np.select(
        [(swing_lows["Low"] < swing_lows["Low"].shift(1)) &
         (swing_lows["Low"] < swing_lows["Low"].shift(-1))],
        ["Intermediate Swing Low"], default='')

    swing_highs["Intermediate_Swing_High"] = np.select(
        [(swing_highs["High"] > swing_highs["High"].shift(1)) &
         (swing_highs["High"] > swing_highs["High"].shift(-1))],
        ["Intermediate Swing High"], default='')

    BTC.loc[swing_lows.index,  "Intermediate_Swing"] = swing_lows["Intermediate_Swing_Low"]
    BTC.loc[swing_lows.index,  "ISL_Price"] = np.where(
        swing_lows["Intermediate_Swing_Low"] == "Intermediate Swing Low", swing_lows["Low"], np.nan)
    BTC.loc[swing_highs.index, "Intermediate_Swing"] = swing_highs["Intermediate_Swing_High"]
    BTC.loc[swing_highs.index, "ISH_Price"] = np.where(
        swing_highs["Intermediate_Swing_High"] == "Intermediate Swing High", swing_highs["High"], np.nan)

    BTC["ISH_Price_Ffill"] = BTC["ISH_Price"].ffill()
    BTC["ISL_Price_Ffill"] = BTC["ISL_Price"].ffill()

    # ── BOS Signal ────────────────────────────────────────────────────────────
    lookback = 3
    Subset_df["Rolling_SL_Min"] = swing_lows["Low"].shift(1).rolling(window=lookback).min()
    Subset_df["Rolling_SH_Max"] = swing_highs["High"].shift(1).rolling(window=lookback).max()

    Subset_df["Bearish BOS"] = np.where(
        (Subset_df["Master_Swing"] == "Swing Low") & (Subset_df["Low"] < Subset_df["Rolling_SL_Min"]),
        True, False)
    Subset_df["Bullish BOS"] = np.where(
        (Subset_df["Master_Swing"] == "Swing High") & (Subset_df["High"] > Subset_df["Rolling_SH_Max"]),
        True, False)

    BTC.loc[Subset_df.index, "Bearish BOS"]    = Subset_df["Bearish BOS"]
    BTC.loc[Subset_df.index, "Bullish BOS"]    = Subset_df["Bullish BOS"]
    # Map rolling reference levels back to BTC so we can filter by ISL position
    BTC.loc[Subset_df.index, "Rolling_SL_Min"] = Subset_df["Rolling_SL_Min"]
    BTC.loc[Subset_df.index, "Rolling_SH_Max"] = Subset_df["Rolling_SH_Max"]

    # ── ISL/ISH Sweeps + BOS confluence ──────────────────────────────────────
    BTC["ISL Sweep"] = ((BTC["Close"].shift(1) > BTC["ISL_Price_Ffill"].shift(1)) &
                        (BTC["Low"] < BTC["ISL_Price_Ffill"].shift(1)))
    BTC["ISH Sweep"] = ((BTC["Close"].shift(1) < BTC["ISH_Price_Ffill"].shift(1)) &
                        (BTC["High"] > BTC["ISH_Price_Ffill"].shift(1)))

    # ISL_Sweep+BOS is valid ONLY when the prior swing lows being broken (Rolling_SL_Min)
    # were themselves ABOVE the ISL level. This ensures we only capture the FIRST break
    # below the ISL — not subsequent BOS events where price is already below the ISL.
    # Example fix: if SL1 already broke the ISL, a later SL2 breaking SL1 is just a
    # continuation move, not a new liquidity sweep.
    BTC["ISL_Sweep+BOS"] = (
        (BTC["Bearish BOS"] == True) &
        (BTC["Low"] < BTC["ISL_Price_Ffill"].shift(1)) &
        (BTC["Rolling_SL_Min"] > BTC["ISL_Price_Ffill"].shift(1))
    )
    BTC["ISH_Sweep+BOS"] = ((BTC["Bullish BOS"] == True) &
                             (BTC["High"] > BTC["ISH_Price_Ffill"].shift(1)))

    # ── FVG ───────────────────────────────────────────────────────────────────
    BTC["FVG_Bearish"] = BTC["High"].shift(-1) < BTC["Low"].shift(1)
    BTC["FVG_Bullish"] = BTC["Low"].shift(-1)  > BTC["High"].shift(1)

    # ── Check signal at position ──────────────────────────────────────────────
    if BTC["ISL_Sweep+BOS"].iloc[position] != True:
        continue

    signal_count += 1
    bos_idx   = BTC.index[position]
    bos_iloc  = BTC.index.get_loc(bos_idx)
    bos_low   = BTC["Low"].iloc[position]
    isl_level = BTC["ISL_Price_Ffill"].iloc[bos_iloc - 1]   # ISL active on candle before BOS

    # ── Preceding Swing High = top of the downswing leg ──────────────────────
    swings_before = Subset_df[Subset_df.index < bos_idx]
    prev_sh_rows  = swings_before[swings_before["Master_Swing"] == "Swing High"]

    if len(prev_sh_rows) == 0:
        Alert_List.append((Ticker, 0.0, 0.0, 0.0, float('nan'), 0.0, bos_idx))
        continue

    sh_idx  = prev_sh_rows.index[-1]
    sh_iloc = BTC.index.get_loc(sh_idx)
    sh_high = BTC.loc[sh_idx, "High"]

    # ── ATR baseline: ATR_BASELINE candles ending AT the swing high ───────────
    # Retrospective — no downswing data is included, so the drop cannot skew it.
    b_start      = max(0, sh_iloc - ATR_BASELINE + 1)
    atr_baseline = mean_atr(BTC.iloc[b_start : sh_iloc + 1])

    # ── Downswing ATR: mean ATR from swing high → BOS candle ─────────────────
    ds_slice      = BTC.iloc[sh_iloc : bos_iloc + 1]
    atr_downswing = mean_atr(ds_slice) if len(ds_slice) > 1 else atr_baseline
    atr_ratio     = round(atr_downswing / atr_baseline, 3) if atr_baseline > 0 else 0.0

    # ── Sweep depth ───────────────────────────────────────────────────────────
    leg_drop    = sh_high - bos_low
    depth_ratio = round(leg_drop / atr_baseline, 3) if atr_baseline > 0 else 0.0

    # ── Bearish FVGs on the downswing leg ─────────────────────────────────────
    # FVG_Bearish at candle i means: High[i+1] < Low[i-1]  →  gap = Low[i-1] - High[i+1]
    fvg_window = BTC[
        (BTC.index >= sh_idx) &
        (BTC.index <= bos_idx) &
        (BTC["FVG_Bearish"] == True)
    ]
    fvg_size = 0.0
    for fvg_ts in fvg_window.index:
        fi = BTC.index.get_loc(fvg_ts)
        if fi == 0 or fi >= len(BTC) - 1:
            continue
        gap = BTC.iloc[fi - 1]["Low"] - BTC.iloc[fi + 1]["High"]
        fvg_size += max(gap, 0.0)

    fvg_atr = round(fvg_size / atr_baseline, 3) if atr_baseline > 0 else 0.0

    # ── Score: depth × energy × (1 + cleanliness) ────────────────────────────
    # (1 + fvg_atr) ensures sweeps without FVGs still score; FVGs multiply the score.
    score = round(depth_ratio * atr_ratio * (1 + fvg_atr), 4)

    # ── ISL buffer (display only) ─────────────────────────────────────────────
    # % distance from the last swing low that was ABOVE the ISL level to the ISL itself.
    # A large value means price had more buffer above the ISL before it was swept —
    # i.e., the sweep punched through a meaningful gap, not a marginal one.
    sl_above_isl = swing_lows[(swing_lows.index < bos_idx) & (swing_lows["Low"] > isl_level)]
    if len(sl_above_isl) > 0 and isl_level > 0:
        isl_buffer_pct = round((sl_above_isl["Low"].iloc[-1] - isl_level) / isl_level * 100, 2)
    else:
        isl_buffer_pct = float('nan')

    Alert_List.append((Ticker, depth_ratio, atr_ratio, fvg_atr, isl_buffer_pct, score, bos_idx))

# ── Rank by score descending ──────────────────────────────────────────────────
Alert_List.sort(key=lambda x: x[5], reverse=True)

print(f"Scan complete — tickers: {scan_count} | ISL_Sweep+BOS at position {position}: {signal_count}")

# ── Send alert ────────────────────────────────────────────────────────────────
if Alert_List:
    alert_time     = Alert_List[0][6]
    alert_time_ist = alert_time.tz_convert('Asia/Kolkata') if hasattr(alert_time, 'tz_convert') else alert_time

    ranked_lines = []
    for rank, (ticker, dep, ar, fa, ib, sc, _) in enumerate(Alert_List, start=1):
        buf_str = f"{ib:.2f}%" if ib == ib else "n/a"
        ranked_lines.append(
            f"{rank}. {ticker}  "
            f"(depth: {dep:.3f} | atr_r: {ar:.3f} | fvg/atr: {fa:.3f} | ISL_buf: {buf_str} | score: {sc:.4f})"
        )

    message = (
        "ISL Sweep+BOS signals\n\n"
        f"Candle: {position}\n"
        f"Time: {alert_time_ist}\n\n"
        + "\n".join(ranked_lines)
    )
    telegram_alert(message)
    print(message)
else:
    no_signal_msg = f"ISL Sweep+BOS Scanner ran — no signals at candle {position}."
    telegram_alert(no_signal_msg)
    print(no_signal_msg)


Scan complete — tickers: 179 | ISL_Sweep+BOS at position -265: 1
ISL Sweep+BOS signals

Candle: -265
Time: 2026-02-24 13:15:00+05:30

1. HINDCOPPER.NS  (depth: 1.859 | atr_r: 0.797 | fvg/atr: 0.000 | ISL_buf: 0.01% | score: 1.4816)
